# Quantum teleportation — dynamic circuits with classical feedback

Mid-circuit measurement plus `if` corrections, authored in plain OpenQASM 3. QVM automatically runs per-shot trajectories when a circuit is dynamic.


In [ ]:
from qvm.qasm3_parser import OpenQASM3Parser
from qvm.simulator import Simulator

THETA = 0.8   # unknown state Ry(theta)|0>
TELEPORT = f"""
OPENQASM 3.0;
include "stdgates.inc";
qubit[3] q;
bit[2] mid;
bit[1] out;
ry({THETA}) q[0];
h q[1];
cx q[1], q[2];
cx q[0], q[1];
h q[0];
mid[0] = measure q[0];
mid[1] = measure q[1];
if (mid[1] == 1) {{ x q[2]; }}
if (mid[0] == 1) {{ z q[2]; }}
out[0] = measure q[2];
"""
teleport = OpenQASM3Parser().parse(TELEPORT)
counts = Simulator().sample(teleport, shots=5000, seed=7)

In [ ]:
import math
p_one = sum(c for bits, c in counts.items() if bits[-1] == "1") / 5000
theory = math.sin(THETA / 2) ** 2
print(f"P(measure 1 on qubit 2) = {p_one:.4f}   theory = {theory:.4f}")
assert abs(p_one - theory) < 0.03
print("The state was teleported intact.")